<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== CELL 1: KẾT NỐI DRIVE VÀ ĐỌC 5 DATASET =====

from google.colab import drive  # Import công cụ kết nối Google Drive với Google Colab.
import os  # Import thư viện os để xử lý đường dẫn.
import pandas as pd  # Import pandas để đọc và xử lý dữ liệu bảng.
import numpy as np  # Import numpy để xử lý dữ liệu số.

drive.mount('/content/drive')  # Gắn Google Drive vào môi trường Colab.

DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'  # Khai báo thư mục gốc của project.

RAW_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'raw')  # Tạo đường dẫn đến thư mục chứa dataset raw.

PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')  # Tạo đường dẫn đến thư mục lưu dataset đã preprocessing.

DATA_FILES = ['Data1.csv', 'Data2.csv', 'Data3.csv', 'Data4.csv', 'Data5.csv']  # Khai báo tên 5 dataset.

INPUT_COLUMNS = ['ud', 'uq', 'Id', 'Iq', 'nm']  # Khai báo 5 biến input của mô hình.

TARGET_COLUMNS = ['Rs', 'Ld', 'Lq']  # Khai báo 3 tham số cần nhận dạng.

TIME_COLUMN = 'Time'  # Khai báo cột thời gian được giữ lại cho PINN.

TRAIN_FILES = ['Data1.csv', 'Data2.csv', 'Data3.csv']  # Khai báo 3 dataset dùng để train #thêm dataset6

VALIDATION_FILES = ['Data4.csv']  # Khai báo dataset dùng để validation.

TEST_FILES = ['Data5.csv']  # Khai báo dataset dùng để test.

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)  # Tạo thư mục processed nếu thư mục chưa tồn tại.

datasets = {}  # Tạo dictionary để lưu 5 DataFrame.

for file_name in DATA_FILES:  # Lặp qua từng dataset.
    file_path = os.path.join(RAW_DATA_DIR, file_name)  # Tạo đường dẫn đầy đủ đến file CSV.
    datasets[file_name] = pd.read_csv(file_path)  # Đọc file CSV vào DataFrame.

print('Đã đọc:', len(datasets), 'dataset')  # In số lượng dataset đã đọc.

print('Thư mục processed:', PROCESSED_DATA_DIR)  # In đường dẫn thư mục sẽ lưu dữ liệu đã xử lý.

Mounted at /content/drive
Đã đọc: 5 dataset
Thư mục processed: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN/dataset/processed


In [2]:
# ===== CELL 2: TÁCH TRAIN / VALIDATION / TEST =====

train_df = pd.concat([datasets[file_name] for file_name in TRAIN_FILES], ignore_index=True)  # Ghép Data1, Data2 và Data3 thành tập Train.

validation_df = pd.concat([datasets[file_name] for file_name in VALIDATION_FILES], ignore_index=True)  # Tạo tập Validation từ Data4.

test_df = pd.concat([datasets[file_name] for file_name in TEST_FILES], ignore_index=True)  # Tạo tập Test từ Data5.

print('===== KÍCH THƯỚC DATASET =====')  # In tiêu đề kiểm tra kích thước.

print('Train:', train_df.shape)  # In kích thước tập Train.

print('Validation:', validation_df.shape)  # In kích thước tập Validation.

print('Test:', test_df.shape)  # In kích thước tập Test.

print('\nTổng số mẫu:', len(train_df) + len(validation_df) + len(test_df))  # In tổng số mẫu của cả ba tập.

===== KÍCH THƯỚC DATASET =====
Train: (900000, 17)
Validation: (300000, 17)
Test: (300000, 17)

Tổng số mẫu: 1500000


In [3]:
# ===== CELL 3: LỌC TRÙNG LẶP PWM, TÁCH INPUT/TARGET VÀ CỜ XÁC LẬP CHUẨN =====

def process_single_dataset(df, di_thresh=0.02, dn_thresh=0.2):
    """
    Hàm làm sạch dữ liệu mô phỏng cho từng file CSV:
    1. Loại bỏ các dòng trùng lặp do lưu dữ liệu nhanh hơn chu kỳ cập nhật của bộ điều khiển FOC.
    2. Tính độ biến thiên thực giữa các chu kỳ điều khiển để gắn cờ xác lập (Steady Mask) chính xác.
    """
    # Kiểm tra xem tại mỗi dòng, giá trị Id, Iq hoặc ud có thực sự cập nhật mới so với dòng trước hay không
    is_new_sample = (df['Id'].diff().abs() > 1e-7) | (df['Iq'].diff().abs() > 1e-7) | (df['ud'].diff().abs() > 1e-7)
    is_new_sample.iloc[0] = True  # Luôn giữ lại dòng đầu tiên của mỗi file

    # Lọc bỏ các dòng bị giữ nguyên giá trị (Zero-Order Hold) và đánh lại chỉ số dòng từ 0
    df_clean = df[is_new_sample].copy().reset_index(drop=True)

    # Tính độ biến thiên dòng điện và tốc độ giữa 2 chu kỳ điều khiển liên tiếp
    d_id = df_clean['Id'].diff().abs().fillna(999)
    d_iq = df_clean['Iq'].diff().abs().fillna(999)
    d_nm = df_clean['nm'].diff().abs().fillna(999)

    # Gắn cờ 1.0 cho vùng xác lập thực sự (biến thiên dòng < 0.02 A và biến thiên tốc độ < 0.2 rpm)
    is_steady = (d_id < di_thresh) & (d_iq < di_thresh) & (d_nm < dn_thresh)
    df_clean['Steady'] = is_steady.astype(np.float32)

    return df_clean

# Áp dụng hàm lọc trùng lặp và gắn cờ xác lập riêng cho từng file trong 5 file CSV
clean_datasets = {f: process_single_dataset(datasets[f]) for f in DATA_FILES}

# Ghép lại thành 3 tập Train, Validation và Test sạch
train_df = pd.concat([clean_datasets[f] for f in TRAIN_FILES], ignore_index=True)
validation_df = pd.concat([clean_datasets[f] for f in VALIDATION_FILES], ignore_index=True)
test_df = pd.concat([clean_datasets[f] for f in TEST_FILES], ignore_index=True)

# Tách Input [ud, uq, Id, Iq, nm], Target [Rs, Ld, Lq] và mảng cờ Steady cho tập Train
X_train = train_df[INPUT_COLUMNS].copy()
Y_train = train_df[TARGET_COLUMNS].copy()
Steady_train = train_df['Steady'].to_numpy(dtype=np.float32)

# Tách Input, Target và mảng cờ Steady cho tập Validation
X_validation = validation_df[INPUT_COLUMNS].copy()
Y_validation = validation_df[TARGET_COLUMNS].copy()
Steady_validation = validation_df['Steady'].to_numpy(dtype=np.float32)

# Tách Input, Target và mảng cờ Steady cho tập Test
X_test = test_df[INPUT_COLUMNS].copy()
Y_test = test_df[TARGET_COLUMNS].copy()
Steady_test = test_df['Steady'].to_numpy(dtype=np.float32)

# Giữ lại cột thời gian Time cho cả 3 tập
Time_train = train_df[[TIME_COLUMN]].copy()
Time_validation = validation_df[[TIME_COLUMN]].copy()
Time_test = test_df[[TIME_COLUMN]].copy()

# In thống kê số lượng mẫu thực tế sau khi loại bỏ trùng lặp PWM
print(f'Số mẫu sau khi lọc trùng lặp PWM -> Train: {len(X_train)} | Val: {len(X_validation)} | Test: {len(X_test)}')
print(f'Tỷ lệ mẫu xác lập chuẩn trong Train: {Steady_train.mean()*100:.1f}%')

Số mẫu sau khi lọc trùng lặp PWM -> Train: 89974 | Val: 29990 | Test: 29995
Tỷ lệ mẫu xác lập chuẩn trong Train: 70.7%


In [4]:
# ===== CELL 4: KIỂM TRA DỮ LIỆU SAU KHI TÁCH FEATURE =====

for name, X, Y in [('Train', X_train, Y_train), ('Validation', X_validation, Y_validation), ('Test', X_test, Y_test)]:  # Lặp qua ba tập dữ liệu.

    print(f'\n===== {name} =====')  # In tên tập dữ liệu hiện tại.

    print('NaN trong Input:', X.isna().sum().sum())  # Đếm NaN trong Input.

    print('NaN trong Target:', Y.isna().sum().sum())  # Đếm NaN trong Target.

    print('Inf trong Input:', np.isinf(X.to_numpy()).sum())  # Đếm Inf trong Input.

    print('Inf trong Target:', np.isinf(Y.to_numpy()).sum())  # Đếm Inf trong Target.

    print('Input min:')  # In tiêu đề giá trị nhỏ nhất của Input.

    print(X.min())  # In giá trị nhỏ nhất của từng Input.

    print('Input max:')  # In tiêu đề giá trị lớn nhất của Input.

    print(X.max())  # In giá trị lớn nhất của từng Input.

    print('Target min:')  # In tiêu đề giá trị nhỏ nhất của Target.

    print(Y.min())  # In giá trị nhỏ nhất của từng Target.

    print('Target max:')  # In tiêu đề giá trị lớn nhất của Target.

    print(Y.max())  # In giá trị lớn nhất của từng Target.


===== Train =====
NaN trong Input: 0
NaN trong Target: 0
Inf trong Input: 0
Inf trong Target: 0
Input min:
ud   -161.153778
uq      0.000000
Id    -50.104235
Iq    -76.821998
nm      0.000000
dtype: float64
Input max:
ud      88.822941
uq     208.673671
Id       4.240509
Iq      80.803500
nm    1007.106284
dtype: float64
Target min:
Rs    0.033828
Ld    0.001125
Lq    0.001266
dtype: float64
Target max:
Rs    0.044186
Ld    0.001332
Lq    0.001438
dtype: float64

===== Validation =====
NaN trong Input: 0
NaN trong Target: 0
Inf trong Input: 0
Inf trong Target: 0
Input min:
ud   -158.604968
uq      0.000000
Id    -32.530302
Iq     -0.220759
nm      0.000000
dtype: float64
Input max:
ud       3.736362
uq     195.278410
Id       3.516850
Iq      80.496980
nm    1002.537272
dtype: float64
Target min:
Rs    0.034475
Ld    0.001126
Lq    0.001267
dtype: float64
Target max:
Rs    0.044183
Ld    0.001309
Lq    0.001437
dtype: float64

===== Test =====
NaN trong Input: 0
NaN trong Target: 0
In

In [5]:
# ===== CELL 5: CHUẨN HÓA INPUT VÀ TARGET =====

from sklearn.preprocessing import StandardScaler  # Import StandardScaler để chuẩn hóa dữ liệu.

X_scaler = StandardScaler()  # Tạo scaler cho 5 biến Input.

Y_scaler = StandardScaler()  # Tạo scaler cho 3 biến Target.

X_train_scaled = X_scaler.fit_transform(X_train)  # Fit scaler bằng Input Train và chuẩn hóa Input Train.

X_validation_scaled = X_scaler.transform(X_validation)  # Dùng scaler của Train để chuẩn hóa Input Validation.

X_test_scaled = X_scaler.transform(X_test)  # Dùng scaler của Train để chuẩn hóa Input Test.

Y_train_scaled = Y_scaler.fit_transform(Y_train)  # Fit scaler bằng Target Train và chuẩn hóa Target Train.

Y_validation_scaled = Y_scaler.transform(Y_validation)  # Dùng scaler của Train để chuẩn hóa Target Validation.

Y_test_scaled = Y_scaler.transform(Y_test)  # Dùng scaler của Train để chuẩn hóa Target Test.

print('X_train_scaled:', X_train_scaled.shape)  # In kích thước Input Train sau chuẩn hóa.

print('X_validation_scaled:', X_validation_scaled.shape)  # In kích thước Input Validation sau chuẩn hóa.

print('X_test_scaled:', X_test_scaled.shape)  # In kích thước Input Test sau chuẩn hóa.

print('Y_train_scaled:', Y_train_scaled.shape)  # In kích thước Target Train sau chuẩn hóa.

print('Y_validation_scaled:', Y_validation_scaled.shape)  # In kích thước Target Validation sau chuẩn hóa.

print('Y_test_scaled:', Y_test_scaled.shape)  # In kích thước Target Test sau chuẩn hóa.

X_train_scaled: (89974, 5)
X_validation_scaled: (29990, 5)
X_test_scaled: (29995, 5)
Y_train_scaled: (89974, 3)
Y_validation_scaled: (29990, 3)
Y_test_scaled: (29995, 3)


In [6]:
# ===== CELL 6: KIỂM TRA KẾT QUẢ CHUẨN HÓA =====

X_train_check = pd.DataFrame(X_train_scaled, columns=INPUT_COLUMNS)  # Chuyển Input Train đã chuẩn hóa thành DataFrame để dễ kiểm tra.

Y_train_check = pd.DataFrame(Y_train_scaled, columns=TARGET_COLUMNS)  # Chuyển Target Train đã chuẩn hóa thành DataFrame để dễ kiểm tra.

print('===== INPUT TRAIN SAU CHUẨN HÓA =====')  # In tiêu đề kiểm tra Input.

print(X_train_check.describe().T)  # In thống kê Input Train sau chuẩn hóa.

print('\n===== TARGET TRAIN SAU CHUẨN HÓA =====')  # In tiêu đề kiểm tra Target.

print(Y_train_check.describe().T)  # In thống kê Target Train sau chuẩn hóa.

===== INPUT TRAIN SAU CHUẨN HÓA =====
      count          mean       std       min       25%       50%       75%  \
ud  89974.0  1.162468e-16  1.000006 -7.603031 -0.092924  0.417085  0.617353   
uq  89974.0 -1.010842e-17  1.000006 -1.251772 -1.077480 -0.275495  1.067306   
Id  89974.0  4.043367e-17  1.000006 -4.046380  0.481817  0.525118  0.530836   
Iq  89974.0  5.054208e-17  1.000006 -4.645362 -0.813830 -0.139989  0.411310   
nm  89974.0  1.971141e-16  1.000006 -1.154577 -1.017147 -0.330764  0.858815   

         max  
ud  5.250782  
uq  1.807076  
Id  0.911893  
Iq  2.750257  
nm  1.613557  

===== TARGET TRAIN SAU CHUẨN HÓA =====
      count          mean       std       min       25%       50%       75%  \
Rs  89974.0  5.256377e-16  1.000006 -1.559093 -0.954135 -0.037057  0.985931   
Ld  89974.0  1.142251e-15  1.000006 -0.917128 -0.532544 -0.521007 -0.459162   
Lq  89974.0 -6.171189e-15  1.000006 -3.507043  0.101578  0.378798  0.442592   

         max  
Rs  1.574112  
Ld  2.3117

In [7]:
# ===== CELL 7: LƯU TOÀN BỘ DATASET ĐÃ PREPROCESSING, STEADY MASK VÀ SCALER =====
import joblib  # Import thư viện joblib để lưu các đối tượng Scaler ra file .pkl

# Lưu mảng Input đã chuẩn hóa của 3 tập Train, Validation, Test dưới định dạng .npy
np.save(os.path.join(PROCESSED_DATA_DIR, 'X_train.npy'), X_train_scaled)
np.save(os.path.join(PROCESSED_DATA_DIR, 'Y_train.npy'), Y_train_scaled)

np.save(os.path.join(PROCESSED_DATA_DIR, 'X_validation.npy'), X_validation_scaled)
np.save(os.path.join(PROCESSED_DATA_DIR, 'Y_validation.npy'), Y_validation_scaled)

np.save(os.path.join(PROCESSED_DATA_DIR, 'X_test.npy'), X_test_scaled)
np.save(os.path.join(PROCESSED_DATA_DIR, 'Y_test.npy'), Y_test_scaled)

# Lưu mảng chuỗi thời gian Time của 3 tập dữ liệu
np.save(os.path.join(PROCESSED_DATA_DIR, 'Time_train.npy'), Time_train.to_numpy())
np.save(os.path.join(PROCESSED_DATA_DIR, 'Time_validation.npy'), Time_validation.to_numpy())
np.save(os.path.join(PROCESSED_DATA_DIR, 'Time_test.npy'), Time_test.to_numpy())

# Lưu mảng cờ đánh dấu trạng thái xác lập (Steady Mask) phục vụ tính Physics Loss cho PINN
np.save(os.path.join(PROCESSED_DATA_DIR, 'Steady_train.npy'), Steady_train)
np.save(os.path.join(PROCESSED_DATA_DIR, 'Steady_validation.npy'), Steady_validation)
np.save(os.path.join(PROCESSED_DATA_DIR, 'Steady_test.npy'), Steady_test)

# Lưu lại bộ chuẩn hóa X_scaler và Y_scaler để giải chuẩn hóa khi tính vật lý và đánh giá Test
joblib.dump(X_scaler, os.path.join(PROCESSED_DATA_DIR, 'X_scaler.pkl'))
joblib.dump(Y_scaler, os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

print('Đã lưu toàn bộ dữ liệu sạch, Steady_mask và Scaler vào:', PROCESSED_DATA_DIR)

Đã lưu toàn bộ dữ liệu sạch, Steady_mask và Scaler vào: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN/dataset/processed
